## Init

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import to_date, avg, max, min, sum, round as spark_round

## Read from silver table

In [0]:
df_silver_read = spark.read.table("weather.weather_hourly")

In [0]:
df_silver_read.display()

## Aggregation 1: Daily summary by city

In [0]:
from pyspark.sql.functions import avg, max, min, sum, round as spark_round, to_date, col

df_gold = (
    df_silver_read
    .withColumn("ingested_date", to_date(col("ingested_timestamp")))
    .groupBy("ingested_date", "state_code")
    .agg(
        spark_round(avg("temperature_celsius"), 2).alias("avg_temperature"),
        spark_round(max("temperature_celsius"), 2).alias("max_temperature"),
        spark_round(min("temperature_celsius"), 2).alias("min_temperature"),
        spark_round(avg("humidity_percentage"), 2).alias("avg_humidity"),
        spark_round(sum("precipitation_mm"), 2).alias("total_precipitation_mm")
    )
)

In [0]:
df_gold.display()

## Write in gold table 

In [0]:
# A. Initialize the gold table in Delta Lake if it is the first time it is being executed 
(
    df_gold.write
        .mode("ignore")
        .format("delta")
        .saveAsTable("weather.daily_city_summary")
)

In [0]:
# # B. Execute MERGE using (city_name + date) as unique key 
# target_table = DeltaTable.forName(spark, "weather.gold.daily_city_summary")

# target_table.alias("target").merge(
#     df_gold_daily.alias("source"),
#     """
#     target.city_name = source.city_name AND
#     target.date = source.date 
#     """
# ).whenMatchedUpdateAll() \
#  .whenNotMatchedInsertAll() \
#  .execute()